In [1]:
# %%
!python3 -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 30.5 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
import warnings
warnings.filterwarnings("ignore")
from spacy.tokens import DocBin
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('../../data/processed/10k_data_modified.csv', encoding='utf-8', encoding_errors='ignore')

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [4]:
df_train.head()

,Ingredient_Phrases,Labels
3012,1 1/2 ounces godiva white chocolate liqueur,"['QUANTITY', 'QUANTITY', 'UNIT', 'O', 'FORM', ..."
4047,wedges lemon,"['FORM', 'NAME']"
9120,1 green onion -LRB- chopped fine -RRB-,"['QUANTITY', 'FORM', 'NAME', 'O', 'O', 'O', 'O']"
10561,2 kg potatoes,"['QUANTITY', 'UNIT', 'NAME']"
5751,"1 ounce black olives , sliced","['QUANTITY', 'UNIT', 'FORM', 'NAME', 'O', 'STA..."


In [5]:
df_train.shape

(8627, 2)

In [5]:
# Step 1: Ensure all entries in 'Ingredient_Phrases' are strings
df_train["Ingredient_Phrases"] = df_train["Ingredient_Phrases"].astype(str)

# Step 2: Remove any empty or None values
df_train = df_train[df_train["Ingredient_Phrases"].str.strip() != '']  # Remove empty strings
df_train = df_train.dropna(subset=["Ingredient_Phrases"])  # Remove None or NaN values

In [7]:
print(df_train.dtypes)

Ingredient_Phrases    object
Labels                object
dtype: object


In [8]:
df_train.head()

,Ingredient_Phrases,Labels
3012,1 1/2 ounces godiva white chocolate liqueur,"['QUANTITY', 'QUANTITY', 'UNIT', 'O', 'FORM', ..."
4047,wedges lemon,"['FORM', 'NAME']"
9120,1 green onion -LRB- chopped fine -RRB-,"['QUANTITY', 'FORM', 'NAME', 'O', 'O', 'O', 'O']"
10561,2 kg potatoes,"['QUANTITY', 'UNIT', 'NAME']"
5751,"1 ounce black olives , sliced","['QUANTITY', 'UNIT', 'FORM', 'NAME', 'O', 'STA..."


In [6]:
import pandas as pd

def get_entity_tuple(df, label_map):
    list_of_entities = []
    
    for index, row in df.iterrows():
        ingredient_phrase = row["Ingredient_Phrases"]
        labels = row["Labels"]
        
        # Handle NaN or missing values
        if pd.isna(ingredient_phrase) or pd.isna(labels):
            continue  
        
        # Convert labels from string format to list format (if they are stored as strings)
        if isinstance(labels, str):
            labels = eval(labels)  # Use eval cautiously if data is safe
        
        # Tokenize by splitting on spaces
        words = ingredient_phrase.split()
        
        # Ensure the number of words matches the number of labels
        if len(words) != len(labels):
            print(f"Skipping row {index} due to mismatch in words ({len(words)}) and labels ({len(labels)})")
            continue
        
        # Extract entity spans
        entities = []
        start_idx = 0
        
        for word, label in zip(words, labels):
            end_idx = start_idx + len(word)
            mapped_label = label_map.get(label, "O")  # Default to 'O' if not in label_map
            
            if mapped_label != "O":  # Ignore 'O' labels
                entities.append((start_idx, end_idx, mapped_label))
            
            start_idx = end_idx + 1  # Account for space
        
        list_of_entities.append((ingredient_phrase, {"entities": entities}))  # Store the result
    
    return list_of_entities

# Example usage with label_map
label_map = {
    'QUANTITY': 'QUANTITY',
    'UNIT': 'UNIT',
    'NAME': 'NAME',
    'FORM': 'FORM',
    'STATE': 'STATE',
    'DF': 'DF',
    'SIZE': 'SIZE',
    'O': 'O'  # Outside token
}

# Call the function safely
train_entities = get_entity_tuple(df_train, label_map)
test_entities = get_entity_tuple(df_test, label_map)

In [10]:
print(train_entities)

[('1 1/2 ounces godiva white chocolate liqueur', {'entities': [(0, 1, 'QUANTITY'), (2, 5, 'QUANTITY'), (6, 12, 'UNIT'), (20, 25, 'FORM'), (26, 35, 'FORM'), (36, 43, 'NAME')]}), ('wedges lemon', {'entities': [(0, 6, 'FORM'), (7, 12, 'NAME')]}), ('1 green onion -LRB- chopped fine -RRB-', {'entities': [(0, 1, 'QUANTITY'), (2, 7, 'FORM'), (8, 13, 'NAME')]}), ('2 kg potatoes', {'entities': [(0, 1, 'QUANTITY'), (2, 4, 'UNIT'), (5, 13, 'NAME')]}), ('1 ounce black olives , sliced', {'entities': [(0, 1, 'QUANTITY'), (2, 7, 'UNIT'), (8, 13, 'FORM'), (14, 20, 'NAME'), (23, 29, 'STATE')]}), ('1 lb ground beef -LRB- very lean -RRB-', {'entities': [(0, 1, 'QUANTITY'), (2, 4, 'UNIT'), (5, 11, 'STATE'), (12, 16, 'NAME')]}), ('10 -15 grape tomatoes , halved', {'entities': [(0, 2, 'QUANTITY'), (3, 6, 'QUANTITY'), (7, 12, 'NAME'), (13, 21, 'NAME'), (24, 30, 'STATE')]}), ('2 large bell peppers , cored seeded , and finely chopped', {'entities': [(0, 1, 'QUANTITY'), (2, 7, 'SIZE'), (8, 12, 'NAME'), (13, 20,

In [7]:
db = DocBin()
nlp = spacy.load("en_core_web_trf")

for text, annot in tqdm(train_entities):  # data in previous format
    doc = nlp.make_doc(text)  # create doc object from text
    ents = []
    for start, end, label in annot["entities"]:  # add character indexes
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            print("Skipping entity")
        else:
            ents.append(span)
    doc.ents = ents  # label the text with the ents
    db.add(doc)
    
db.to_disk("../../data/spacy/train.spacy")

100%|██████████| 8627/8627 [00:00<00:00, 10664.69it/s]


In [13]:
# from tqdm import tqdm
# import spacy
# # from spacy.training import DocBin

# db = DocBin()
# nlp = spacy.load("en_core_web_trf")

# for text, annot in tqdm(train_entities):  # data in previous format
#     doc = nlp.make_doc(text)  # create doc object from text
#     ents = []
#     for start, end, label in annot["entities"]:  # add character indexes
#         # Change alignment_mode to "expand" for better handling of entities
#         span = doc.char_span(start, end, label=label, alignment_mode="expand")
#         if span is None:
#             print(f"Skipping entity: {text[start:end]}")  # Log problematic span
#         else:
#             ents.append(span)
#     doc.ents = ents  # label the text with the ents
#     db.add(doc)
    
# db.to_disk("../../data/spacy/train.spacy")


100%|██████████| 7000/7000 [00:00<00:00, 7221.28it/s]


In [8]:
db = DocBin()
nlp = spacy.load("en_core_web_trf")

for text, annot in tqdm(test_entities):  # data in previous format
    doc = nlp.make_doc(text)  # create doc object from text
    ents = []
    for start, end, label in annot["entities"]:  # add character indexes
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            print("Skipping entity")
        else:
            ents.append(span)
    doc.ents = ents  # label the text with the ents
    db.add(doc)

db.to_disk("../../data/spacy/test.spacy")  # save the docbin object

100%|██████████| 2157/2157 [00:00<00:00, 8885.33it/s]


In [8]:
# import os

# # Setting the CUDA and library paths
# os.environ['PATH'] = '/usr/local/cuda-11.6/bin:' + os.environ['PATH']
# os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.6/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')


In [ ]:
import locale
def getpreferredencoding(do_setlocale=True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding
!python3 -m spacy train ../../configs/spacy/config.cfg --output ../../models/spacy_deberta_v3_large --paths.train ../../data/spacy/train.spacy --paths.dev ../../data/spacy/test.spacy --gpu-id 1 

ℹ Saving to output directory: output
ℹ Using GPU: 1

=========================== Initializing pipeline ===========================
/home/ritisha21089/miniconda3/lib/python3.9/site-packages/thinc/shims/pytorch.py:261: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded fi

In [13]:
import locale
def getpreferredencoding(do_setlocale=True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding
!python3 -m spacy train ../../configs/spacy/config.cfg --output ../../models/spacy_deberta_v3_large --paths.train ../../data/spacy/train.spacy --paths.dev ../../data/spacy/test.spacy --gpu-id 1 

ℹ Saving to output directory: output
ℹ Using GPU: 1

=========================== Initializing pipeline ===========================
/home/ritisha21089/miniconda3/lib/python3.9/site-packages/thinc/shims/pytorch.py:261: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded fi

In [13]:
import locale
def getpreferredencoding(do_setlocale=True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding
!python3 -m spacy train ../../configs/spacy/config.cfg --output ../../models/spacy_deberta_v3_large --paths.train ../../data/spacy/train.spacy --paths.dev ../../data/spacy/test.spacy --gpu-id 1 


✔ Created output directory: output
ℹ Saving to output directory: output
ℹ Using GPU: 1

=========================== Initializing pipeline ===========================
/home/ritisha21089/miniconda3/lib/python3.9/site-packages/thinc/shims/pytorch.py:261: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't

In [15]:
import spacy
from spacy.training import Example
from sklearn.metrics import classification_report


# Load the trained model
nlp = spacy.load("../../models/spacy_deberta_v3_large/model-best")  # Adjust this path to your trained model's directory

def evaluate_model(nlp, test_entities, label_map):
    true_labels = []
    pred_labels = []

    # Map labels to integers for consistency with classification report
    label_to_id = {label: idx for idx, label in enumerate(label_map)}
    labels = list(label_to_id.values())  # List of label IDs for classification_report

    for text, annotation in test_entities:
        # Process the document with the model
        doc = nlp(text)
        example = Example.from_dict(doc, annotation)

        # Extract true entities from the example and predicted entities from the model
        true_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.reference.ents]
        pred_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.predicted.ents]

        # Token-wise alignment: generate token-level labels
        for token in doc:
            true_label = "O"  # Default to "O" if no entity
            pred_label = "O"  # Default to "O" if no entity

            # Check if the token is part of a true entity
            for start, end, label in true_ents:
                if token.idx >= start and token.idx < end:
                    true_label = label
                    break

            # Check if the token is part of a predicted entity
            for start, end, label in pred_ents:
                if token.idx >= start and token.idx < end:
                    pred_label = label
                    break

            # Convert labels to integer IDs
            true_labels.append(label_to_id.get(true_label, label_to_id["O"]))
            pred_labels.append(label_to_id.get(pred_label, label_to_id["O"]))

    # Generate classification report
    print("Classification Report:")
    print(classification_report(true_labels, pred_labels, labels=labels, target_names=label_map.keys(), zero_division=1))

# Define the label map, ensure it matches your data
label_map = {
    'QUANTITY': 'QUANTITY',
    'UNIT': 'UNIT',
    'NAME': 'NAME',
    'FORM': 'FORM',
    'STATE': 'STATE',
    'DF': 'DF',
    'SIZE': 'SIZE',
    'O': 'O'  # Outside token
}

# Run the evaluation function
evaluate_model(nlp, test_entities, label_map)

Classification Report:
              precision    recall  f1-score   support

    QUANTITY       0.98      0.98      0.98      2454
        UNIT       0.98      0.98      0.98      1710
        NAME       0.94      0.95      0.94      3464
        FORM       0.91      0.86      0.88       827
       STATE       0.94      0.93      0.94      2106
          DF       0.95      0.97      0.96       186
        SIZE       0.84      0.91      0.88       449
           O       0.95      0.94      0.94      6370

    accuracy                           0.95     17566
   macro avg       0.94      0.94      0.94     17566
weighted avg       0.95      0.95      0.95     17566



In [15]:
import spacy
from spacy.training import Example
from sklearn.metrics import classification_report


# Load the trained model
nlp = spacy.load("../../models/spacy_deberta_v3_large/model-best")  # Adjust this path to your trained model's directory

def evaluate_model(nlp, test_entities, label_map):
    true_labels = []
    pred_labels = []

    # Map labels to integers for consistency with classification report
    label_to_id = {label: idx for idx, label in enumerate(label_map)}
    labels = list(label_to_id.values())  # List of label IDs for classification_report

    for text, annotation in test_entities:
        # Process the document with the model
        doc = nlp(text)
        example = Example.from_dict(doc, annotation)

        # Extract true entities from the example and predicted entities from the model
        true_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.reference.ents]
        pred_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.predicted.ents]

        # Token-wise alignment: generate token-level labels
        for token in doc:
            true_label = "O"  # Default to "O" if no entity
            pred_label = "O"  # Default to "O" if no entity

            # Check if the token is part of a true entity
            for start, end, label in true_ents:
                if token.idx >= start and token.idx < end:
                    true_label = label
                    break

            # Check if the token is part of a predicted entity
            for start, end, label in pred_ents:
                if token.idx >= start and token.idx < end:
                    pred_label = label
                    break

            # Convert labels to integer IDs
            true_labels.append(label_to_id.get(true_label, label_to_id["O"]))
            pred_labels.append(label_to_id.get(pred_label, label_to_id["O"]))

    # Generate classification report
    print("Classification Report:")
    print(classification_report(true_labels, pred_labels, labels=labels, target_names=label_map.keys(), zero_division=1))

# Define the label map, ensure it matches your data
label_map = {
    'QUANTITY': 'QUANTITY',
    'UNIT': 'UNIT',
    'NAME': 'NAME',
    'FORM': 'FORM',
    'STATE': 'STATE',
    'DF': 'DF',
    'SIZE': 'SIZE',
    'O': 'O'  # Outside token
}

# Run the evaluation function
evaluate_model(nlp, test_entities, label_map)

Classification Report:
              precision    recall  f1-score   support

    QUANTITY       0.98      0.98      0.98      2454
        UNIT       0.98      0.98      0.98      1710
        NAME       0.94      0.95      0.94      3464
        FORM       0.91      0.86      0.88       827
       STATE       0.94      0.93      0.94      2106
          DF       0.95      0.97      0.96       186
        SIZE       0.84      0.91      0.88       449
           O       0.95      0.94      0.94      6370

    accuracy                           0.95     17566
   macro avg       0.94      0.94      0.94     17566
weighted avg       0.95      0.95      0.95     17566



- evalaution w/o "O" tag

In [14]:
import spacy
from spacy.training import Example
from sklearn.metrics import classification_report, f1_score

# Load the trained model
nlp = spacy.load("../../models/spacy_deberta_v3_large/model-best")  # Adjust this path to your trained model's directory

def evaluate_model(nlp, test_entities, label_map):
    true_labels = []
    pred_labels = []

    # Map labels to integers for consistency with classification report
    label_to_id = {label: idx for idx, label in enumerate(label_map)}
    
    # Filter out "O" for evaluation
    filtered_labels = [label for label in label_map if label != "O"]
    filtered_label_ids = [label_to_id[label] for label in filtered_labels]

    for text, annotation in test_entities:
        doc = nlp(text)
        example = Example.from_dict(doc, annotation)

        true_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.reference.ents]
        pred_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.predicted.ents]

        for token in doc:
            true_label = "O"
            pred_label = "O"

            for start, end, label in true_ents:
                if start <= token.idx < end:
                    true_label = label
                    break

            for start, end, label in pred_ents:
                if start <= token.idx < end:
                    pred_label = label
                    break

            true_labels.append(label_to_id.get(true_label, label_to_id["O"]))
            pred_labels.append(label_to_id.get(pred_label, label_to_id["O"]))

    # Generate classification report without "O"
    print("Classification Report (excluding 'O'):")
    print(classification_report(
        true_labels, 
        pred_labels, 
        labels=filtered_label_ids, 
        target_names=filtered_labels, 
        zero_division=1
    ))

    macro_f1 = f1_score(true_labels, pred_labels, labels=filtered_label_ids, average='macro', zero_division=1)
    print(f"\nMacro F1 Score (excluding 'O'): {macro_f1:.4f}")

# Define the label map
label_map = {
    'QUANTITY': 'QUANTITY',
    'UNIT': 'UNIT',
    'NAME': 'NAME',
    'FORM': 'FORM',
    'STATE': 'STATE',
    'DF': 'DF',
    'SIZE': 'SIZE',
    'O': 'O'  # Outside token
}

# Run the evaluation function
evaluate_model(nlp, test_entities, label_map)

Classification Report (excluding 'O'):
              precision    recall  f1-score   support

    QUANTITY       0.98      0.98      0.98      2454
        UNIT       0.98      0.98      0.98      1710
        NAME       0.94      0.95      0.94      3464
        FORM       0.91      0.86      0.88       827
       STATE       0.94      0.93      0.94      2106
          DF       0.95      0.97      0.96       186
        SIZE       0.84      0.91      0.88       449

   micro avg       0.95      0.95      0.95     11196
   macro avg       0.93      0.94      0.94     11196
weighted avg       0.95      0.95      0.95     11196


Macro F1 Score (excluding 'O'): 0.9366
